# Knuth-Morris-Pratt (KMP) Algorithm

The KMP algorithm is one of the most elegant string matching algorithms, achieving **O(n + m)** time complexity by avoiding redundant comparisons. This notebook covers:

1. **The Key Insight** - Why naive string matching wastes work
2. **Prefix Function (Failure Function)** - The heart of KMP
3. **Building the Prefix Table** - Naive O(n²) vs Optimized O(n)
4. **KMP Algorithm** - Using the prefix table for efficient search

---

[< Previous: Naive Pattern Matching Algorithm](01_naive_pattern_matching.ipynb) | [Tier 4: Algorithms & Data Structures Overview](../README.md) | [Next: Rabin-Karp Algorithm >](03_rabin_karp.ipynb)

---

## 1. The Key Insight: Don't Restart from Scratch

### Why Does the Naive Algorithm Waste Work?

In naive string matching, after a mismatch, we shift the pattern by 1 and restart comparison from the beginning. But this throws away valuable information!

```
Text:    A B A B A B A C A B A
Pattern: A B A B A C A
         ✓ ✓ ✓ ✓ ✓ ✗     <- mismatch at position 5

Naive approach: Shift pattern by 1, restart from scratch
Text:    A B A B A B A C A B A
Pattern:   A B A B A C A
           ✗               <- immediately fails!

We already knew that text[2..4] = "ABA" because we just matched it!
Why compare again?
```

### The Smart Observation

When we have a mismatch after matching some characters, we know:
- What characters we just matched in the text
- These characters are part of the pattern

**Key Question:** After matching `P[0..j-1]`, can we find a shorter prefix of the pattern that would still match the end of what we just saw?

```
Pattern matched so far: A B A B A
                        ↑───↑       This prefix "ABA"...
                            ↑───↑   ...equals this suffix!

So after mismatch, we can continue from position 3 in the pattern!
We don't need to re-match "ABA" - we know it's already there.
```

## 2. The Prefix Function (Failure Function)

### Definition

For a pattern `P` of length `m`, the **prefix function** π is defined as:

$$\pi[i] = \text{length of the longest proper prefix of } P[0..i] \text{ that is also a suffix}$$

**Proper prefix:** A prefix that is not the entire string.

### Example: Pattern "ABABACA"

```
Pattern: A B A B A C A

Index:   0   1   2   3   4   5   6
Char:    A   B   A   B   A   C   A
π[i]:    0   0   1   2   3   0   1
```

Let's verify each value:

```
i=0: P[0..0] = "A"
     No proper prefix exists → π[0] = 0

i=1: P[0..1] = "AB"
     Proper prefixes: "A"
     Suffixes: "B"
     No match → π[1] = 0

i=2: P[0..2] = "ABA"
     Proper prefixes: "A", "AB"
     Suffixes: "A", "BA"
     Match: "A" (length 1) → π[2] = 1

i=3: P[0..3] = "ABAB"
     Proper prefixes: "A", "AB", "ABA"
     Suffixes: "B", "AB", "BAB"
     Match: "AB" (length 2) → π[3] = 2

i=4: P[0..4] = "ABABA"
     Proper prefixes: "A", "AB", "ABA", "ABAB"
     Suffixes: "A", "BA", "ABA", "BABA"
     Match: "ABA" (length 3) → π[4] = 3

i=5: P[0..5] = "ABABAC"
     No proper prefix equals any suffix → π[5] = 0

i=6: P[0..6] = "ABABACA"
     Match: "A" (length 1) → π[6] = 1
```

### Visual Explanation for π[4] = 3

```
P[0..4] = "ABABA"
Longest proper prefix that is also suffix = "ABA" (length 3)

A B A B A
↑─────↑      prefix  P[0..2]
    ↑─────↑  suffix  P[2..4]
    
Both are "ABA" → π[4] = 3
```

## 3. Building the Prefix Table

### 3.1 Naive Approach - O(n²)

For each position, check all possible prefix lengths:

In [ ]:
def build_prefix_naive(pattern: str) -> list[int]:
    """
    Build prefix function using naive O(n²) approach.
    
    For each position i, checks all possible prefix lengths from longest
    to shortest until finding one that matches the suffix.
    
    Args:
        pattern: The pattern string to build prefix table for
        
    Returns:
        List where pi[i] = length of longest proper prefix of pattern[0..i]
        that is also a suffix
        
    Time Complexity: O(n²) where n = len(pattern)
    Space Complexity: O(n) for the prefix table
    """
    n = len(pattern)
    pi = [0] * n
    
    for i in range(1, n):
        # Try all possible prefix lengths from longest to shortest
        for length in range(i, 0, -1):
            prefix = pattern[0:length]
            suffix = pattern[i - length + 1:i + 1]
            if prefix == suffix:
                pi[i] = length
                break
    
    return pi


# Test
pattern = "ABABACA"
print(f"Pattern: {pattern}")
print(f"Prefix table (naive): {build_prefix_naive(pattern)}")

### 3.2 Optimized Approach - O(n)

The key insight: **use previously computed values to avoid redundant work!**

If we know π[i-1] = k, then for position i:
- We already have P[0..k-1] = P[i-k..i-1]
- Just check if P[k] == P[i]
- If yes: π[i] = k + 1
- If no: fall back to π[k-1] and try again

### Step-by-Step Example

```
Pattern: A B A B A C A
         0 1 2 3 4 5 6

Initialize: pi[0] = 0, j = 0

i=1: Compare P[1]='B' with P[0]='A'
     B ≠ A, j=0 can't fall back
     → π[1] = 0

i=2: Compare P[2]='A' with P[0]='A'
     A = A → j becomes 1
     → π[2] = 1

i=3: Compare P[3]='B' with P[1]='B'
     B = B → j becomes 2
     → π[3] = 2

i=4: Compare P[4]='A' with P[2]='A'
     A = A → j becomes 3
     → π[4] = 3

i=5: Compare P[5]='C' with P[3]='B'
     C ≠ B → fall back: j = π[2] = 1
     Compare P[5]='C' with P[1]='B'
     C ≠ B → fall back: j = π[0] = 0
     Compare P[5]='C' with P[0]='A'
     C ≠ A, j=0 can't fall back
     → π[5] = 0

i=6: Compare P[6]='A' with P[0]='A'
     A = A → j becomes 1
     → π[6] = 1

Final: [0, 0, 1, 2, 3, 0, 1]
```

In [ ]:
def build_prefix(pattern: str) -> list[int]:
    """
    Build prefix function using optimized O(n) approach.
    
    Uses the key insight that if π[i-1] = k, then we only need to check
    if pattern[k] == pattern[i]. If not, we fall back using previously
    computed π values.
    
    Args:
        pattern: The pattern string to build prefix table for
        
    Returns:
        List where pi[i] = length of longest proper prefix of pattern[0..i]
        that is also a suffix
        
    Time Complexity: O(n) where n = len(pattern)
    Space Complexity: O(n) for the prefix table
    """
    n = len(pattern)
    if n == 0:
        return []
    
    pi = [0] * n
    j = 0  # length of previous longest prefix-suffix
    
    for i in range(1, n):
        # Fall back until we find a match or reach the beginning
        while j > 0 and pattern[j] != pattern[i]:
            j = pi[j - 1]
        
        # If characters match, extend the prefix-suffix
        if pattern[j] == pattern[i]:
            j += 1
        
        pi[i] = j
    
    return pi


# Test and verify
pattern = "ABABACA"
print(f"Pattern: {pattern}")
print(f"Prefix table (optimized): {build_prefix(pattern)}")
print(f"Prefix table (naive):     {build_prefix_naive(pattern)}")
assert build_prefix(pattern) == build_prefix_naive(pattern), "Results should match!"

### Why is the Optimized Version O(n)?

It looks like there's a nested loop (while inside for), suggesting O(n²). But consider:

- `j` can only increase by 1 per iteration of the outer loop (at most n increases)
- `j` decreases in the while loop, but can never go below 0
- Total decreases ≤ total increases ≤ n

Therefore, the total number of operations is **O(n)**.

This is called **amortized analysis** - while individual iterations may do more work, the total work is bounded.

In [ ]:
def build_prefix_verbose(pattern: str) -> list[int]:
    """
    Build prefix function with step-by-step output for learning.
    
    Args:
        pattern: The pattern string
        
    Returns:
        The prefix table
    """
    n = len(pattern)
    if n == 0:
        return []
    
    pi = [0] * n
    j = 0
    
    print(f"Pattern: {pattern}")
    print(f"         {''.join(str(i) for i in range(n))}")
    print(f"\nBuilding prefix table:\n")
    print(f"pi[0] = 0 (by definition)\n")
    
    for i in range(1, n):
        print(f"i={i}: Checking P[{i}]='{pattern[i]}'")
        
        while j > 0 and pattern[j] != pattern[i]:
            print(f"      P[{j}]='{pattern[j]}' ≠ P[{i}]='{pattern[i]}' → fall back to j=pi[{j-1}]={pi[j-1]}")
            j = pi[j - 1]
        
        if pattern[j] == pattern[i]:
            j += 1
            print(f"      P[{j-1}]='{pattern[j-1]}' = P[{i}]='{pattern[i]}' → extend to j={j}")
        else:
            print(f"      P[{j}]='{pattern[j]}' ≠ P[{i}]='{pattern[i]}', j=0 → no extension")
        
        pi[i] = j
        print(f"      → pi[{i}] = {j}\n")
    
    print(f"Final prefix table: {pi}")
    return pi


# Demonstrate with verbose output
_ = build_prefix_verbose("ABABACA")

## 4. The KMP Algorithm

### How KMP Uses the Prefix Function

When searching for pattern `P` in text `T`:
1. If `T[i] == P[j]`: both advance (match)
2. If `T[i] != P[j]` and `j > 0`: use prefix function to shift pattern
   - Set `j = π[j-1]` (the pattern slides so the prefix matches what we've seen)
3. If `T[i] != P[j]` and `j == 0`: move to next character in text

### Visual Example

```
Text:    A B A B A B A C A B A
Pattern: A B A B A C A
π:       [0,0,1,2,3,0,1]

Step 1-5: Match first 5 characters (positions 0-4)
Text:    A B A B A B A C A B A
         ↓ ↓ ↓ ↓ ↓ ✗
Pattern: A B A B A C A
         0 1 2 3 4 5
                   ^ mismatch at j=5 (P[5]='C' ≠ T[5]='B')

Instead of restarting, use π[4] = 3:
- We know P[0..2] = P[2..4] = "ABA"
- The text characters T[2..4] = "ABA" (we just matched them!)
- So P[0..2] already matches T[2..4]
- Jump: j = π[j-1] = π[4] = 3

Text:    A B A B A B A C A B A
               ↓ ↓ ↓ ↓ ↓ ↓ ↓
Pattern:       A B A B A C A
               0 1 2 3 4 5 6
                     ^ continue from j=3, i stays at 5

Text index never goes backward! Always O(n) for text traversal.
```

In [ ]:
def kmp_search(text: str, pattern: str) -> list[int]:
    """
    Find all occurrences of pattern in text using KMP algorithm.
    
    The KMP algorithm preprocesses the pattern to build a prefix table,
    then uses it to avoid redundant comparisons during search. The text
    pointer never moves backward.
    
    Args:
        text: The text to search in
        pattern: The pattern to search for
        
    Returns:
        List of starting indices where pattern occurs in text
        
    Time Complexity: O(n + m) where n = len(text), m = len(pattern)
    Space Complexity: O(m) for the prefix table
    """
    if not pattern:
        return []
    if len(pattern) > len(text):
        return []
    
    # Build prefix table
    pi = build_prefix(pattern)
    
    matches = []
    j = 0  # position in pattern
    
    for i in range(len(text)):
        # On mismatch, use prefix table to shift pattern
        while j > 0 and text[i] != pattern[j]:
            j = pi[j - 1]
        
        # If characters match, advance in pattern
        if text[i] == pattern[j]:
            j += 1
        
        # If complete pattern matched
        if j == len(pattern):
            matches.append(i - j + 1)
            j = pi[j - 1]  # Look for next occurrence
    
    return matches


# Test examples
print("Example 1:")
text1 = "ABABABABACA"
pattern1 = "ABABACA"
result1 = kmp_search(text1, pattern1)
print(f"Text:    '{text1}'")
print(f"Pattern: '{pattern1}'")
print(f"Found at positions: {result1}")

print("\nExample 2 (overlapping matches):")
text2 = "AAAAAA"
pattern2 = "AAA"
result2 = kmp_search(text2, pattern2)
print(f"Text:    '{text2}'")
print(f"Pattern: '{pattern2}'")
print(f"Found at positions: {result2}")

print("\nExample 3:")
text3 = "ABCABCABCAB"
pattern3 = "ABCABCAB"
result3 = kmp_search(text3, pattern3)
print(f"Text:    '{text3}'")
print(f"Pattern: '{pattern3}'")
print(f"Found at positions: {result3}")

In [ ]:
def kmp_search_verbose(text: str, pattern: str) -> list[int]:
    """
    KMP search with detailed step-by-step output.
    
    Args:
        text: The text to search in
        pattern: The pattern to search for
        
    Returns:
        List of starting indices where pattern occurs
    """
    if not pattern or len(pattern) > len(text):
        return []
    
    pi = build_prefix(pattern)
    print(f"Text:    '{text}'")
    print(f"Pattern: '{pattern}'")
    print(f"Prefix:  {pi}")
    print("\n" + "="*50 + "\n")
    
    matches = []
    j = 0
    comparisons = 0
    
    for i in range(len(text)):
        while j > 0 and text[i] != pattern[j]:
            comparisons += 1
            print(f"i={i}, j={j}: T[{i}]='{text[i]}' ≠ P[{j}]='{pattern[j]}' → shift: j=pi[{j-1}]={pi[j-1]}")
            j = pi[j - 1]
        
        comparisons += 1
        if text[i] == pattern[j]:
            print(f"i={i}, j={j}: T[{i}]='{text[i]}' = P[{j}]='{pattern[j]}' → match, j++")
            j += 1
        else:
            print(f"i={i}, j={j}: T[{i}]='{text[i]}' ≠ P[{j}]='{pattern[j]}' → no match, move to next i")
        
        if j == len(pattern):
            pos = i - j + 1
            matches.append(pos)
            print(f"  *** FOUND at position {pos}! ***")
            j = pi[j - 1]
    
    print(f"\nTotal comparisons: {comparisons}")
    return matches


# Demonstrate
print("KMP Search Step-by-Step:")
print()
_ = kmp_search_verbose("ABABABABACA", "ABABACA")

## 5. Complexity Analysis

| Aspect | Complexity | Explanation |
|--------|------------|-------------|
| **Prefix table construction** | O(m) | Each character processed once, j increases at most m times |
| **Search phase** | O(n) | Text pointer i never goes backward, j increases at most n times |
| **Total time** | O(n + m) | Sum of preprocessing and search |
| **Space** | O(m) | For storing the prefix table |

### Comparison with Naive Algorithm

| Algorithm | Time | Space |
|-----------|------|-------|
| Naive | O(n × m) worst case | O(1) |
| KMP | O(n + m) | O(m) |

## 6. Performance Comparison

Let's compare KMP with naive string matching on worst-case inputs:

In [ ]:
def naive_search_with_count(text: str, pattern: str) -> tuple[list[int], int]:
    """
    Naive string matching with comparison counter.
    
    Args:
        text: Text to search in
        pattern: Pattern to search for
        
    Returns:
        Tuple of (list of match positions, comparison count)
    """
    matches = []
    comparisons = 0
    n, m = len(text), len(pattern)
    
    for i in range(n - m + 1):
        match = True
        for j in range(m):
            comparisons += 1
            if text[i + j] != pattern[j]:
                match = False
                break
        if match:
            matches.append(i)
    
    return matches, comparisons


def kmp_search_with_count(text: str, pattern: str) -> tuple[list[int], int]:
    """
    KMP search with comparison counter.
    
    Args:
        text: Text to search in
        pattern: Pattern to search for
        
    Returns:
        Tuple of (list of match positions, comparison count)
    """
    if not pattern or len(pattern) > len(text):
        return [], 0
    
    pi = build_prefix(pattern)
    matches = []
    j = 0
    comparisons = 0
    
    for i in range(len(text)):
        while j > 0 and text[i] != pattern[j]:
            comparisons += 1
            j = pi[j - 1]
        
        comparisons += 1
        if text[i] == pattern[j]:
            j += 1
        
        if j == len(pattern):
            matches.append(i - j + 1)
            j = pi[j - 1]
    
    return matches, comparisons

In [ ]:
# Worst case for naive: text = "AAAA...AAB", pattern = "AAA...AAB"
# The pattern almost matches at every position!

print("WORST CASE COMPARISON")
print("="*60)
print()

for size in [100, 500, 1000, 2000]:
    # Construct worst case
    text = "A" * size + "B"
    pattern = "A" * (size // 10) + "B"
    
    naive_matches, naive_comps = naive_search_with_count(text, pattern)
    kmp_matches, kmp_comps = kmp_search_with_count(text, pattern)
    
    # Verify same results
    assert naive_matches == kmp_matches, "Results should match!"
    
    ratio = naive_comps / kmp_comps if kmp_comps > 0 else 0
    
    print(f"Text length: {len(text)}, Pattern length: {len(pattern)}")
    print(f"  Naive comparisons: {naive_comps:,}")
    print(f"  KMP comparisons:   {kmp_comps:,}")
    print(f"  KMP is {ratio:.1f}x faster")
    print(f"  Matches found: {len(kmp_matches)}")
    print()

In [ ]:
# Another worst case: many overlapping patterns
print("\nOVERLAPPING PATTERNS CASE")
print("="*60)
print()

for size in [1000, 5000, 10000]:
    text = "A" * size
    pattern = "A" * 10
    
    naive_matches, naive_comps = naive_search_with_count(text, pattern)
    kmp_matches, kmp_comps = kmp_search_with_count(text, pattern)
    
    assert naive_matches == kmp_matches
    
    ratio = naive_comps / kmp_comps if kmp_comps > 0 else 0
    
    print(f"Text: '{len(text)} A's', Pattern: '10 A's'")
    print(f"  Matches found: {len(kmp_matches)}")
    print(f"  Naive comparisons: {naive_comps:,}")
    print(f"  KMP comparisons:   {kmp_comps:,}")
    print(f"  KMP is {ratio:.1f}x faster")
    print()

## 7. Practical Example: DNA Sequence Matching

A common application of string matching is finding gene sequences in DNA:

In [ ]:
import random

def generate_dna_sequence(length: int) -> str:
    """Generate a random DNA sequence."""
    return ''.join(random.choice('ACGT') for _ in range(length))


def find_gene_occurrences(genome: str, gene: str) -> dict:
    """
    Find all occurrences of a gene in a genome.
    
    Args:
        genome: The DNA sequence to search in
        gene: The gene pattern to search for
        
    Returns:
        Dictionary with search results and statistics
    """
    positions = kmp_search(genome, gene)
    return {
        'gene': gene,
        'genome_length': len(genome),
        'occurrences': len(positions),
        'positions': positions[:10],  # First 10 positions
        'prefix_table': build_prefix(gene)
    }


# Example: Find a specific pattern in DNA
random.seed(42)
genome = generate_dna_sequence(10000)

# Search for a specific gene pattern
gene1 = "ACGTACGT"
result1 = find_gene_occurrences(genome, gene1)

print("DNA SEQUENCE MATCHING")
print("="*50)
print(f"\nGenome length: {result1['genome_length']} base pairs")
print(f"Gene pattern: {result1['gene']}")
print(f"Prefix table: {result1['prefix_table']}")
print(f"Occurrences found: {result1['occurrences']}")
if result1['positions']:
    print(f"First positions: {result1['positions']}")

# Search for a repetitive pattern (common in DNA)
gene2 = "ATATAT"  # Alternating pattern
result2 = find_gene_occurrences(genome, gene2)

print(f"\nGene pattern: {result2['gene']}")
print(f"Prefix table: {result2['prefix_table']}")
print(f"Occurrences found: {result2['occurrences']}")

In [ ]:
# Real-world example: Find start codon (ATG) in a sequence
print("\nFINDING START CODONS (ATG)")
print("="*50)

# Simulate a realistic coding sequence
coding_sequence = "ATGGCGATGCGTACGATGATGCTAGATGACG"
start_codon = "ATG"

positions = kmp_search(coding_sequence, start_codon)

print(f"Sequence: {coding_sequence}")
print(f"Start codon (ATG) found at positions: {positions}")

# Visualize the matches
print("\nVisualization:")
print(f"  {coding_sequence}")
marker = [' '] * len(coding_sequence)
for pos in positions:
    marker[pos] = '^'
print(f"  {''.join(marker)}")

## 8. Edge Cases and Testing

In [ ]:
def test_kmp():
    """Comprehensive tests for KMP algorithm."""
    tests = [
        # (text, pattern, expected_positions)
        ("", "ABC", []),                           # Empty text
        ("ABC", "", []),                           # Empty pattern
        ("ABC", "ABCD", []),                       # Pattern longer than text
        ("ABC", "ABC", [0]),                       # Exact match
        ("ABCABC", "ABC", [0, 3]),                 # Multiple matches
        ("AAAA", "AA", [0, 1, 2]),                 # Overlapping matches
        ("ABCDEF", "XYZ", []),                     # No match
        ("ABABABABAB", "ABAB", [0, 2, 4, 6]),      # Many overlapping
        ("A", "A", [0]),                           # Single character match
        ("AAAAAB", "AAAB", [2]),                   # Near-miss pattern
    ]
    
    print("Testing KMP Algorithm")
    print("="*50)
    
    all_passed = True
    for text, pattern, expected in tests:
        result = kmp_search(text, pattern)
        status = "✓" if result == expected else "✗"
        if result != expected:
            all_passed = False
        
        text_display = text if len(text) <= 15 else text[:12] + "..."
        print(f"{status} text='{text_display}', pattern='{pattern}'")
        if result != expected:
            print(f"    Expected: {expected}, Got: {result}")
    
    print()
    if all_passed:
        print("All tests passed! ✓")
    else:
        print("Some tests failed! ✗")


test_kmp()

In [ ]:
def test_prefix_function():
    """Test prefix function with known values."""
    tests = [
        # (pattern, expected_prefix_table)
        ("A", [0]),
        ("AB", [0, 0]),
        ("AA", [0, 1]),
        ("AAA", [0, 1, 2]),
        ("ABABACA", [0, 0, 1, 2, 3, 0, 1]),
        ("ABCABC", [0, 0, 0, 1, 2, 3]),
        ("AABAAAB", [0, 1, 0, 1, 2, 2, 3]),
        ("ABCDABCD", [0, 0, 0, 0, 1, 2, 3, 4]),
    ]
    
    print("Testing Prefix Function")
    print("="*50)
    
    all_passed = True
    for pattern, expected in tests:
        result = build_prefix(pattern)
        naive_result = build_prefix_naive(pattern)
        
        optimized_ok = result == expected
        naive_ok = naive_result == expected
        
        if optimized_ok and naive_ok:
            print(f"✓ '{pattern}' → {result}")
        else:
            all_passed = False
            print(f"✗ '{pattern}'")
            print(f"    Expected:  {expected}")
            print(f"    Optimized: {result}")
            print(f"    Naive:     {naive_result}")
    
    print()
    if all_passed:
        print("All tests passed! ✓")


test_prefix_function()

## 9. Summary

### Key Takeaways

1. **The Problem with Naive Search**: After a mismatch, naive search throws away information about characters already matched.

2. **The Prefix Function**: π[i] tells us the length of the longest proper prefix of P[0..i] that is also a suffix. This captures "self-similarity" in the pattern.

3. **How KMP Uses This**: When a mismatch occurs after matching j characters, instead of restarting, we use π[j-1] to know how many characters we can "keep" - they're guaranteed to still match.

4. **The Magic**: The text pointer **never moves backward**. We make at most O(n) comparisons regardless of the pattern.

### When to Use KMP

- When searching for the same pattern multiple times (prefix table can be reused)
- When the pattern has repetitive structure
- When worst-case guarantees matter
- When you can't afford O(nm) worst-case performance

### Complexity Summary

| Operation | Time | Space |
|-----------|------|-------|
| Build prefix table | O(m) | O(m) |
| Search | O(n) | O(1) |
| **Total** | **O(n + m)** | **O(m)** |

## Alternative Implementation (Kodomo)

The Kodomo coursework uses a **j = -1 sentinel** variant of KMP. Instead of checking `j > 0` before following the failure link, the loop condition `while j >= 0` handles both the "still matching" and "not matching" cases uniformly, with `j = -1` acting as a signal to advance without a match.

This alternative uses j = -1 as a sentinel value instead of checking j > 0. Both are correct; the sentinel approach avoids one branch.

The net effect is identical; the sentinel version is slightly more compact at the cost of a less obvious loop invariant.

In [ ]:
def prefix_function_sentinel(p: str) -> list[int]:
    """KMP prefix array using j = -1 as a sentinel."""
    sp = [0] * len(p)
    j = 0
    for i in range(1, len(p)):
        while j >= 0 and p[j] != p[i]:
            j = sp[j - 1] if j - 1 >= 0 else -1
        j += 1
        sp[i] = j
    return sp


def kmp_search_sentinel(text: str, pattern: str) -> list[int]:
    """KMP search using the -1 sentinel variant."""
    matches = []
    f = prefix_function_sentinel(pattern)
    n, m = len(text), len(pattern)
    j = 0
    for i in range(n):
        while j >= 0 and text[i] != pattern[j]:
            j = f[j - 1] if j - 1 >= 0 else -1
        j += 1
        if j == m:
            matches.append(i - m + 1)
            j = f[m - 1]
    return matches


# Verify against a known result
text = "ATGCATGCATGC"
pattern = "ATGC"
print(f"Sentinel KMP: {kmp_search_sentinel(text, pattern)}")  # [0, 4, 8]

# Overlapping case
print(f"Overlapping 'AA' in 'AAAA': {kmp_search_sentinel('AAAA', 'AA')}")  # [0, 1, 2]

---

## Exercises

### Exercise 1: Promoter Motif Search in a Bacterial Genome (*)

The TATA box (consensus: TATAAA) is a promoter element found ~30 bp upstream of transcription start sites in many bacterial genes. Finding all TATA box occurrences in a genome is a direct application of KMP.

**Task:** Use `kmp_search` to find all occurrences of the TATA box motif `"TATAAA"` in the simulated bacterial genome below. Report each position and the surrounding 10-nucleotide context (`sequence[pos-5:pos+11]`).

Extend: also search for the variant `"TATATA"` and report how many positions contain either form.

In [ ]:
import random
random.seed(7)

# Simulated bacterial genome with planted TATA boxes
_base = ''.join(random.choice('ACGT') for _ in range(500))
# Plant TATAAA at positions 50, 150, 300
_genome = list(_base)
for pos in [50, 150, 300]:
    for i, c in enumerate("TATAAA"):
        _genome[pos + i] = c
BACTERIAL_GENOME = ''.join(_genome)

TATA_BOX = "TATAAA"
TATA_VARIANT = "TATATA"


def find_promoter_motifs(genome: str, motif: str) -> list[int]:
    """
    Find all occurrences of a promoter motif in a genome using KMP.

    Args:
        genome: The DNA sequence to search
        motif: The motif pattern (e.g. TATAAA)

    Returns:
        List of start positions where motif occurs
    """
    # YOUR CODE HERE
    pass


# positions = find_promoter_motifs(BACTERIAL_GENOME, TATA_BOX)
# print(f"TATA box '{TATA_BOX}' found {len(positions)} time(s):")
# for pos in positions:
#     context = BACTERIAL_GENOME[max(0, pos-5):pos+11]
#     print(f"  pos {pos:4d}: ...{context}...")
#
# variant_positions = find_promoter_motifs(BACTERIAL_GENOME, TATA_VARIANT)
# all_positions = sorted(set(positions) | set(variant_positions))
# print(f"\nAll TATA-form positions ({len(all_positions)} total): {all_positions}")

### Exercise 2: Shortest Period of a DNA Tandem Repeat (**)

Tandem repeats (e.g., microsatellites like CACACACACA) are repetitive elements important in forensics and population genetics. The KMP failure function reveals the shortest period of a string efficiently.

**Task:** Implement `find_tandem_period(sequence)` using the KMP failure function. The period `p` of a string of length `n` satisfies:
- `failure[n-1] > 0` (the string has some self-overlap)
- `n % (n - failure[n-1]) == 0` (the remainder divides evenly)
- If both hold, `p = n - failure[n-1]`; otherwise `p = n` (the string itself is the period)

Test on the sequences below and verify by checking `sequence == sequence[:p] * (n // p)`.

In [ ]:
TANDEM_SEQUENCES = [
    "CACACACACA",       # period 2: CA
    "ATGATGATG",        # period 3: ATG
    "AGTCAGTCAGTC",    # period 4: AGTC
    "ATCGATCG",         # period 4: ATCG
    "ATCGATCGATCG",    # period 4: ATCG
    "AAAAAAAAAA",       # period 1: A
    "ACGTACGTACGT",    # period 4: ACGT
    "ATCGTTGA",         # no repeat — period = len
]


def find_tandem_period(sequence: str) -> int:
    """
    Find the shortest period of a DNA tandem repeat using the KMP failure function.

    If failure[n-1] > 0 and n % (n - failure[n-1]) == 0,
    the period is n - failure[n-1]. Otherwise the period is n itself.

    Args:
        sequence: DNA string

    Returns:
        Length of the shortest period
    """
    # YOUR CODE HERE
    pass


# print("Tandem repeat period detection:")
# for seq in TANDEM_SEQUENCES:
#     period = find_tandem_period(seq)
#     unit = seq[:period]
#     reps = len(seq) // period
#     verified = (unit * reps == seq[:period * reps])
#     print(f"  '{seq}': period={period} ('{unit}') verified={verified}")

---

[< Previous: Naive Pattern Matching Algorithm](01_naive_pattern_matching.ipynb) | [Tier 4: Algorithms & Data Structures Overview](../README.md) | [Next: Rabin-Karp Algorithm >](03_rabin_karp.ipynb)

---